### Comentario de entrada

Este bloque agrega la carpeta superior al directorio de búsqueda de Python e importa la función `download_csv`. Luego, define la URL del conjunto de datos Mushroom e intenta descargarlo, utilizando una referencia relativa para evitar depender de rutas específicas de una computadora.

In [ ]:
import sys
sys.path.append("..")  # raíz del proyecto para poder importar src/

from src.inf8239_u01.data import download_csv

URL = "https://www.kaggle.com/datasets/uciml/mushroom-classification"
path = download_csv(URL)
print(path)



Si la fuente requiere ZIP, API o autenticación, documenta y encapsula ese procedimiento. No uses C:\Users\... ni /content/drive/... como única ruta.

d:\Roberto Byas\Desktop\8239\INF8239_U01\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 34.2k/34.2k [00:00<00:00, 659kB/s]

Extracting files...


D:\Roberto Byas\Desktop\8239\INF8239_U01\data\raw\dataset.csv


### Comentario de salida

Si la descarga se completa correctamente, se muestra la ruta del archivo obtenido. La URL corresponde a una página de Kaggle, no a un CSV directo; por tanto, la función debe contemplar el procedimiento necesario para descargar los datos.



### Comentario de entrada

Este bloque carga el archivo `dataset.csv` en un DataFrame de pandas mediante una ruta relativa al directorio de trabajo. A continuación, permite reconocer su estructura: cantidad de filas y columnas, tipos de datos y registros iniciales y finales.

In [3]:
import pandas as pd
df = pd.read_csv("../data/raw/dataset.csv")
print(df.shape)
print(df.dtypes)
print(df.head())
print(df.tail())
assert not df.empty

(8124, 23)
class                       str
cap-shape                   str
cap-surface                 str
cap-color                   str
bruises                     str
odor                        str
gill-attachment             str
gill-spacing                str
gill-size                   str
gill-color                  str
stalk-shape                 str
stalk-root                  str
stalk-surface-above-ring    str
stalk-surface-below-ring    str
stalk-color-above-ring      str
stalk-color-below-ring      str
veil-type                   str
veil-color                  str
ring-number                 str
ring-type                   str
spore-print-color           str
population                  str
habitat                     str
dtype: object
  class cap-shape cap-surface cap-color bruises odor gill-attachment  \
0     p         x           s         n       t    p               f   
1     e         x           s         y       t    a               f   
2     e         b      

### Comentario de salida

Se muestran las dimensiones del conjunto de datos, los tipos de datos de cada columna y las primeras y últimas cinco filas. La instrucción `assert not df.empty` verifica que el DataFrame no esté vacío; si lo está, genera un error. Esta revisión inicial no garantiza la ausencia de valores faltantes, duplicados o tipos de datos incorrectos.

### Comentario de entrada

Este bloque construye una auditoría por columna del DataFrame: tipo de dato, cantidad y porcentaje de valores ausentes y número de valores únicos. Ordena las columnas de mayor a menor porcentaje de ausentes y cuenta las filas completamente duplicadas.

In [4]:

audit = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "porcentaje_ausente": (df.isna().mean()*100).round(2),
    "unicos": df.nunique(dropna=False)
}).sort_values("porcentaje_ausente", ascending=False)
print("Duplicados:", df.duplicated().sum())
display(audit)



Duplicados: 0


,tipo,ausentes,porcentaje_ausente,unicos
class,str,0,0.0,2
stalk-surface-above-ring,str,0,0.0,4
population,str,0,0.0,6
spore-print-color,str,0,0.0,9
ring-type,str,0,0.0,5
ring-number,str,0,0.0,3
veil-color,str,0,0.0,4
veil-type,str,0,0.0,1
stalk-color-below-ring,str,0,0.0,9
stalk-color-above-ring,str,0,0.0,9


### Comentario de salida

Se presenta el total de filas duplicadas y una tabla con los indicadores de auditoría. El conteo de valores únicos incluye los valores ausentes, mientras que `isna()` solo reconoce los faltantes identificados como tales por pandas; símbolos como `?` requieren una revisión adicional.

Para completar el diccionario de datos, deben añadirse el significado, la unidad, la fuente, el momento de disponibilidad, la transformación prevista y el riesgo de cada variable, especialmente el de fuga de información. Estos campos requieren consultar la documentación del conjunto de datos y no se generan automáticamente con este código.

### Comentario de entrada

Se define `class` como variable objetivo, cuyos valores son `e` (comestible) y `p` (venenoso). Se mantiene vacía la lista de exclusiones porque no se han justificado columnas adicionales que deban retirarse por su significado o disponibilidad al momento de predecir. La variable objetivo se excluye de los predictores para evitar incorporarla como información de entrada.

In [5]:
TARGET = "class"
DROP_COLUMNS = []  # No se han justificado exclusiones adicionales

assert TARGET in df.columns

X = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET]

print("Clases: e = comestible; p = venenoso")
print(y.value_counts(dropna=False))

assert y.notna().all()
assert y.nunique() >= 2

Clases: e = comestible; p = venenoso
class
e    4208
p    3916
Name: count, dtype: int64


### Comentario de salida

Se obtienen las variables predictoras en `X` y la variable objetivo en `y`. Para el archivo analizado, se muestran 4,208 registros de hongos comestibles y 3,916 de venenosos. Las comprobaciones verifican que el target no tenga valores ausentes y contenga al menos dos clases. Cualquier exclusión posterior deberá justificarse por razones semánticas o de disponibilidad, no únicamente por su efecto sobre la métrica.

### Comentario de entrada

Este bloque separa los predictores de `X` en numéricos y categóricos y define su preprocesamiento. Para los numéricos, establece la imputación de ausentes mediante la mediana y la estandarización. Para los categóricos, utiliza la categoría más frecuente para imputar ausentes y aplica codificación One-Hot, configurada para tolerar categorías desconocidas durante la transformación.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocess = ColumnTransformer([("num", num_pipe, num_cols),
                                ("cat", cat_pipe, cat_cols)])
print(len(num_cols), len(cat_cols))

0 22


### Comentario de salida

Para el conjunto Mushroom cargado, se espera la salida `0 22`: ningún predictor numérico y 22 categóricos. El objeto `preprocess` queda definido, pero todavía no se ajusta ni transforma los datos. Para evitar fugas de información, debe ajustarse únicamente con los datos de entrenamiento y dentro de cada partición de validación cruzada, integrado en el pipeline del modelo.

Los valores representados por `?` deben convertirse previamente en valores ausentes si así lo establece el diccionario de datos; de lo contrario, se tratarán como una categoría adicional.

### Comentario de entrada

Este bloque divide los datos en entrenamiento (80 %) y prueba (20 %), manteniendo aproximadamente la proporción de clases mediante estratificación. Define dos pipelines: un modelo de referencia que siempre predice la clase más frecuente y una SVM con kernel RBF. Ambos incorporan el preprocesamiento, que se ajusta con los datos de entrenamiento. La semilla `42` permite reproducir la partición.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score

Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)
dummy=Pipeline([("prep",preprocess),("model",DummyClassifier(strategy="most_frequent"))])
svm=Pipeline([("prep",preprocess),("model",SVC(C=1,gamma="scale",probability=True,random_state=42))])
for name,model in {"dummy":dummy,"svm":svm}.items():
    model.fit(Xtr,ytr)
    pred=model.predict(Xte)
    print(name, f1_score(yte,pred,average="macro"))
print(classification_report(yte,svm.predict(Xte)))

dummy 0.341305229023105


d:\Roberto Byas\Desktop\8239\INF8239_U01\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


svm 1.0
              precision    recall  f1-score   support

           e       1.00      1.00      1.00       842
           p       1.00      1.00      1.00       783

    accuracy                           1.00      1625
   macro avg       1.00      1.00      1.00      1625
weighted avg       1.00      1.00      1.00      1625



### Comentario de salida

Se muestra el F1 macro de cada modelo sobre el conjunto de prueba, asignando la misma importancia a ambas clases. También se presenta el informe de clasificación de la SVM, con precisión, sensibilidad, F1 y cantidad de casos por clase. Estos resultados permiten comparar la SVM con el modelo de referencia; sus valores deben interpretarse después de ejecutar el código.

Para la clase `p` (venenoso), una sensibilidad baja indica que algunos hongos venenosos se clasifican como comestibles. El conjunto de prueba debe reservarse para la evaluación final y no utilizarse para seleccionar hiperparámetros.